In [1]:
from dataclasses import dataclass

import pymupdf
from tqdm import tqdm
from deep_translator import GoogleTranslator

In [2]:
@dataclass
class Product:
    name: str
    description: str
    specs: dict
    image: bytes

In [3]:
doc = pymupdf.open("raw/2026箭牌卫浴产品-5-9.pdf")

In [4]:
def get_product_image(page) -> bytes | None:

    best = {}
    for img in page.get_images(full=True):

        xref = img[0]

        rects = page.get_image_rects(xref)
        if not rects:
            continue
        
        rect = rects[0]
        
        width = rect.width
        height = rect.height

        area = width * height

        if area > page.rect.width * page.rect.height * 0.5:
            continue

        score = rect.x0 + rect.y0

        if not best or score < best["score"]:
            best = {
                "xref": xref,
                "score": score,
            }
    
    if best:
        return doc.extract_image(best["xref"])

    return {}

In [5]:
def get_product_name(page) -> str | None:

    text_dict = page.get_text("dict")

    best = {}
    for block in text_dict["blocks"]:

        if block["type"] != 0:
            continue

        for line in block["lines"]:

            for span in line["spans"]:

                text = span["text"].strip()

                if not text:
                    continue

                x0, y0, x1, y1 = span["bbox"]

                # Только верх страницы
                if y0 > 150:
                    continue

                size = span["size"]

                score = (
                    size * 1000
                    - x0
                    - y0
                )

                if not best or score > best["score"]:
                    best = {
                        "text": text,
                        "score": score,
                    }
    
    return best.get("text")

In [6]:
for page in tqdm(doc):

    specs = page.search_for("产品规格")
    overview = page.search_for("产品简介")
    release = page.search_for("上市时间：")

    if not specs or not overview or not release:
        continue

    spec_anchor = specs[0]
    overview_anchor = overview[0]
    release_anchor = release[0]

    spec_rect = pymupdf.Rect(
        spec_anchor.x0,
        spec_anchor.y1,
        page.rect.width,
        overview_anchor.y0
    )
    overview_rect = pymupdf.Rect(
        overview_anchor.x0,
        overview_anchor.y1,
        page.rect.width,
        release_anchor.y0,
    )

    specs_text = page.get_textbox(spec_rect)
    overview_text = page.get_textbox(overview_rect)

    product_image = get_product_image(page)
    product_name = get_product_name(page)

    image_bytes = product_image["image"]
    with open(f"images/{product_name}.png", "wb") as f:
        f.write(image_bytes)

  9%|▉         | 15/166 [00:01<00:13, 10.95it/s]


FileNotFoundError: [Errno 2] No such file or directory: 'images/AG1078-1/AD1008-1 连体坐便器-534.png'